# FinSight · Live stock price tool (Finnhub)

**Owner:** P1 (Data)

This notebook builds the tool that answers *"what is NVIDIA trading at right now?"*

| | SEC tools (EDGAR, XBRL) | Price tool (this notebook) |
|---|---|---|
| Data | Filings and reported figures | Live market price |
| Changes | Every quarter | Every second |
| Stored in our SQLite database? | Yes (batch path) | **No**: fetched live, cached 60 s |

**Finnhub free tier:** real-time US quotes, about **60 calls per minute**, free key from [finnhub.io](https://finnhub.io), no credit card.

> This is the exploration version. The same code lives in `src/finsight/tools/prices.py` in the repo; fix bugs there first, then copy here.

## 1. Setup
Get a free key at finnhub.io → Dashboard, and add it to the `.env` file in your Drive:
```
FINNHUB_API_KEY="your_key_here"
```

In [ ]:
# Colab already has these; this is for running locally
#%pip install -q requests pandas matplotlib python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# In Colab: load the key from the .env file in Google Drive (keeps it out of the notebook)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ENV_PATH = "/content/drive/MyDrive/Colab Notebooks/6_CMPE-259/.env"
except ImportError:          # running locally (VS Code / Jupyter): use the repo's .env
    ENV_PATH = ".env"

assert os.path.exists(ENV_PATH), f".env not found at {ENV_PATH}"
load_dotenv(ENV_PATH, override=True)

# Check it loaded, without printing the key
key = os.getenv("FINNHUB_API_KEY", "")
assert key and "your_finnhub_key" not in key, "Add FINNHUB_API_KEY to your .env"
print("FINNHUB_API_KEY loaded:", key[:4] + "…")

## 2. The price tool

Finnhub's `/quote` endpoint returns short field names; the tool turns them into a `Quote`:

| Finnhub | `Quote` field | Meaning |
|---|---|---|
| `c` | `price` | last price |
| `d` / `dp` | `change` / `change_pct` | vs previous close ($ / %) |
| `o`, `h`, `l` | `open`, `high`, `low` | today's range |
| `pc` | `prev_close` | previous close |
| `t` | `as_of` | time of the last price |

**Built-in safeguards**
- **Rate limit:** ≥ 1.1 s between calls, and each quote is reused for 60 s.
- **Market closed:** if the last trade is older than 15 min (nights, weekends), `market_open_likely=False`, so we never present Friday's price as "live".
- **Unknown ticker:** Finnhub returns zeros; we raise an error instead of reporting $0.
- **Key in a header** (`X-Finnhub-Token`), not in the URL, so it doesn't end up in logs.

In [ ]:
"""Live stock price tool (Finnhub free tier).

Answers "what is NVIDIA trading at right now?" for the agents and the UI.

Source: https://finnhub.io/api/v1/quote?symbol=NVDA   (free key from finnhub.io)
  c  = current price      d  = change        dp = change %
  h  = day high           l  = day low       o  = open
  pc = previous close     t  = time of the last price (Unix seconds)

Rules:
  - The key comes from .env (FINNHUB_API_KEY). Never commit it.
  - Free tier is about 60 calls/minute: we wait >= 1.1 s between calls and keep
    each quote for 60 s, so a burst of questions doesn't burn the limit.
  - Prices are NOT stored in the SQLite datastore: they change every second.
    Filings and reported figures (SEC) stay the "batch path"; prices are live.
  - Unknown tickers come back from Finnhub as all zeros; we raise an error
    instead of reporting a price of $0.
"""


import os
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import requests


QUOTE_URL = "https://finnhub.io/api/v1/quote"
MIN_INTERVAL = 1.1        # seconds between calls -> under 60 calls/minute
QUOTE_TTL = 60            # seconds a quote is reused before asking again
STALE_AFTER = 15 * 60     # last trade older than this -> market likely closed

_session = requests.Session()
_last_call = 0.0
_cache: dict[str, tuple[float, "Quote"]] = {}


@dataclass
class Quote:
    ticker: str
    price: float              # last price
    change: float             # vs previous close, in $
    change_pct: float         # vs previous close, in %
    open: float
    high: float
    low: float
    prev_close: float
    as_of: str               # time of the last price, ISO UTC
    market_open_likely: bool  # False if the last price is old (weekend, night)
    source: str = "Finnhub"
    source_url: str = ""

    def to_dict(self) -> dict:
        return asdict(self)

    def summary(self) -> str:
        """One line for chat answers, e.g. 'NVDA $182.40 (+1.25%, as of 2026-10-16 20:00 UTC, Finnhub)'."""
        when = self.as_of.replace("T", " ")[:16] + " UTC"
        status = "" if self.market_open_likely else ", market closed"
        return (f"{self.ticker} ${self.price:,.2f} ({self.change_pct:+.2f}%, "
                f"as of {when}{status}, {self.source})")


def _api_key() -> str:
    key = os.getenv("FINNHUB_API_KEY", "")
    if not key:
        raise RuntimeError("Set FINNHUB_API_KEY in your .env (free key from https://finnhub.io)")
    return key


def _get(params: dict) -> dict:
    """GET /quote with rate limiting and retry on 429 / server errors."""
    global _last_call
    for attempt in range(3):
        wait = MIN_INTERVAL - (time.time() - _last_call)
        if wait > 0:
            time.sleep(wait)
        _last_call = time.time()
        resp = _session.get(QUOTE_URL, params=params,
                            headers={"X-Finnhub-Token": _api_key()}, timeout=15)
        if resp.status_code == 429 or resp.status_code >= 500:
            time.sleep(2 ** attempt * 2)  # 2s, 4s, 8s
            continue
        if resp.status_code in (401, 403):
            raise RuntimeError("Finnhub rejected the key (check FINNHUB_API_KEY), "
                               "or this endpoint isn't on the free plan")
        resp.raise_for_status()
        return resp.json()
    resp.raise_for_status()
    return resp.json()


def parse_quote(ticker: str, data: dict, now: float | None = None) -> Quote:
    """Turn Finnhub's short field names into a Quote. Pure function (unit-tested)."""
    if not data or not data.get("t") or not data.get("c"):
        raise ValueError(f"No price for {ticker!r}: unknown ticker or not a US stock")
    now = now if now is not None else time.time()
    return Quote(
        ticker=ticker.upper(),
        price=float(data["c"]),
        change=float(data.get("d") or 0.0),
        change_pct=float(data.get("dp") or 0.0),
        open=float(data.get("o") or 0.0),
        high=float(data.get("h") or 0.0),
        low=float(data.get("l") or 0.0),
        prev_close=float(data.get("pc") or 0.0),
        as_of=datetime.fromtimestamp(data["t"], tz=timezone.utc).isoformat(timespec="seconds"),
        market_open_likely=(now - data["t"]) < STALE_AFTER,
        source_url=f"https://finnhub.io/quote/{ticker.upper()}",
    )


def get_quote(ticker: str, max_age: int = QUOTE_TTL) -> Quote:
    """Latest price for one ticker, e.g. get_quote("NVDA").price.

    Reuses a quote fetched in the last `max_age` seconds (0 = always ask Finnhub).
    """
    t = ticker.upper().strip()
    hit = _cache.get(t)
    if hit and time.time() - hit[0] < max_age:
        return hit[1]
    quote = parse_quote(t, _get({"symbol": t}))
    _cache[t] = (time.time(), quote)
    return quote


def get_quotes(tickers: list[str], max_age: int = QUOTE_TTL) -> dict[str, Quote | str]:
    """Quotes for several tickers. A failed ticker gets an error message instead of a Quote."""
    out: dict[str, Quote | str] = {}
    for t in tickers:
        try:
            out[t.upper()] = get_quote(t, max_age)
        except Exception as e:  # noqa: BLE001 — one bad ticker shouldn't stop the rest
            out[t.upper()] = f"{type(e).__name__}: {e}"
    return out

## 3. Check the logic offline
Made-up values, no internet, no API calls. The same checks run in the repo's `tests/test_prices.py`.

In [ ]:
SAMPLE = {"c": 182.4, "d": 2.25, "dp": 1.2489, "h": 184.1, "l": 179.8,
          "o": 180.5, "pc": 180.15, "t": 1_760_644_800}  # made-up values

q = parse_quote("nvda", SAMPLE, now=SAMPLE["t"] + 60)          # one minute after the trade
print(q.summary())
assert q.ticker == "NVDA" and q.price == 182.4 and q.market_open_likely

q_weekend = parse_quote("NVDA", SAMPLE, now=SAMPLE["t"] + 3 * 24 * 3600)
print(q_weekend.summary())
assert not q_weekend.market_open_likely

try:
    parse_quote("NOTAREALTICKER", {"c": 0, "t": 0})
except ValueError as e:
    print("Unknown ticker ->", e)

print("\n✓ fields mapped; market-closed flag works; unknown ticker rejected")

## 4. Try it on live data

In [ ]:
import pandas as pd

# Ask the user to enter a company ticker.
TICKER = input("Enter a stock ticker (e.g., NVDA, AAPL, MSFT): ").strip().upper()

q = get_quote(TICKER)
print(q.summary())
pd.Series(q.to_dict())

In [ ]:
# Ask again right away: the answer comes from the 60-second cache, no API call
import time
t0 = time.time()
again = get_quote(TICKER)
print(f"second call took {time.time() - t0:.3f} s, same object: {again is q}")

# max_age=0 forces a fresh call (waits ~1 s for the rate limit)
t0 = time.time()
fresh = get_quote(TICKER, max_age=0)
print(f"fresh call took {time.time() - t0:.2f} s -> {fresh.summary()}")

## 5. All 10 companies
About one call per second, so 10 tickers take roughly 10–15 seconds. A failed ticker shows its error instead of stopping the table.

In [ ]:
COMPANIES = ["NVDA", "AAPL", "MSFT", "AMZN", "GOOGL", "META", "TSLA", "JPM", "AMD", "INTC"]  # ← your 10

rows = []
for t, q in get_quotes(COMPANIES).items():
    if isinstance(q, Quote):
        rows.append({"ticker": t, "price": q.price, "change_%": round(q.change_pct, 2),
                     "prev_close": q.prev_close, "day_low": q.low, "day_high": q.high,
                     "as_of (UTC)": q.as_of, "market_open": q.market_open_likely})
    else:
        rows.append({"ticker": t, "error": q})

prices_df = pd.DataFrame(rows)
prices_df

In [ ]:
import matplotlib.pyplot as plt

ok = prices_df.dropna(subset=["change_%"]).sort_values("change_%")
colors = ["#c0392b" if v < 0 else "#2b6cb0" for v in ok["change_%"]]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(ok["ticker"], ok["change_%"], color=colors)
ax.axvline(0, color="#888", linewidth=0.8)
ax.set_xlabel("Change vs previous close (%)")
ax.set_title("Today's move (Finnhub quotes)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Notes and next steps
- **Context only:** FinSight never gives buy / hold / sell advice. Agents should state the price and its time and source, not turn a move into a recommendation.
- **Cite it:** every `Quote` has `source`, `source_url` and `as_of`, so the Report agent can cite a price like any other number.
- **Historical prices** (for charts over months) likely aren't on Finnhub's free plan; fallbacks from our API list: Alpha Vantage daily prices (25 calls/day, cache them) or yfinance.
- **Combine with XBRL later:** e.g. price ÷ trailing-12-month diluted EPS (from the SEC data) = P/E, computed in code.